## Imports and device setup

In [ ]:
import os
import io
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

import torch
import torch.nn as nn

# FIX: Use float64 everywhere. Computing 4th-order spatial derivatives via
# autograd in float32 accumulates too much floating-point error — each grad()
# call roughly squares the relative error. float64 fixes this.
DTYPE = torch.float64

from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dtype:", DTYPE)

torch.manual_seed(42)
np.random.seed(42)


## Geometry and experiment settings

Set your beam geometry and physical constants here. These are fixed known quantities, not learned by the PINN.

In [ ]:
# Beam geometry
L = 126.06e-3          # beam length [m]
width = 10.13e-3       # width [m]
height = 0.95e-3       # thickness / height [m]
rho = 8216.0           # density [kg/m^3]

bending_axis = "weak"
x_meas = L             # tip measurement location

# Known tip mass
m_tip = 10e-3          # [kg]

# ── Initial guesses for unknowns ──────────────────────────────────────────
# FIX: Initial guesses must be physically plausible for YOUR beam material.
# Wrong guesses (e.g. E0=210GPa for aluminium) put the optimizer in a bad
# landscape — the resonance frequencies predicted by the physics will be
# completely wrong before training even begins.
#
# For steel:     E0 ~ 200e9 Pa
# For aluminium: E0 ~ 70e9 Pa
# For plastics:  E0 ~ 1-5e9 Pa
# eta (damping loss factor): typically 0.001 – 0.05
# kx, kphi: start soft (1e4 – 1e7) unless you have prior knowledge
E0_init    = 200e9     # [Pa]  <- adjust to your material
eta_init   = 0.01      # [-]
kx_init    = 1e6       # [N/m]
kphi_init  = 1e4       # [N·m/rad]
F_init     = 0.1       # [N]

# Synthetic reference values (used only for synthetic validation)
E0_true    = 200e9
eta_true   = 0.01
kx_true    = 1e6
kphi_true  = 1e4
F_true     = 0.1

# ── Frequency cutoff ──────────────────────────────────────────────────────
# FIX: Euler-Bernoulli beam theory is only valid for the first few bending
# modes. Beyond ~5000 Hz for this beam, the experimental data is noise-
# dominated and the physics model is no longer accurate. Training on this
# high-frequency noise actively HARMS parameter identification because the
# data loss pulls parameters toward fitting noise rather than physics.
MAX_FREQ_HZ = 5000.0   # [Hz]  hard cutoff

# Data options
use_synthetic_as_training_data = True   # ALWAYS validate on synthetic first!
save_processed_csv = True
processed_csv_name = "processed_frf_data.csv"

# Plot options
plot_in_db = True
eps_db = 1e-16

# Compute geometry derived quantities
A = width * height
if bending_axis.lower() == "weak":
    I = width * height**3 / 12.0
elif bending_axis.lower() == "strong":
    I = height * width**3 / 12.0
else:
    raise ValueError("bending_axis must be 'weak' or 'strong'")

print(f"L       = {L:.6e} m")
print(f"width   = {width:.6e} m")
print(f"height  = {height:.6e} m")
print(f"rho     = {rho:.6e} kg/m^3")
print(f"A       = {A:.6e} m^2")
print(f"I       = {I:.6e} m^4")
print(f"x_meas  = {x_meas:.6e} m")
print(f"m_tip   = {m_tip:.6e} kg")
print(f"axis    = {bending_axis}")
print(f"MAX_FREQ_HZ = {MAX_FREQ_HZ} Hz")


## Forward model (Euler-Bernoulli beam)

This is the analytical solution for the beam FRF. It is used to:
1. Generate synthetic training data for validation
2. Define what the PINN's physics loss is trying to enforce

In [ ]:
def area_and_inertia(width, height, bending_axis="weak"):
    A = width * height
    if bending_axis.lower() == "weak":
        I = width * height**3 / 12.0
    elif bending_axis.lower() == "strong":
        I = height * width**3 / 12.0
    else:
        raise ValueError("bending_axis must be 'weak' or 'strong'")
    return A, I


def euler_beam_changes_forward(
    f_hz, L, width, height, rho, E0, eta, F_tip,
    m_tip=0.0, k_phi=0.0, k_x=0.0, x_meas=None, bending_axis="weak",
):
    if x_meas is None:
        x_meas = L

    f_hz = np.asarray(f_hz, dtype=float).reshape(-1)
    A, I = area_and_inertia(width, height, bending_axis)

    v = np.zeros(len(f_hz), dtype=np.complex128)
    E_complex = E0 * (1.0 + 1j * eta)

    for idx, f in enumerate(f_hz):
        omega = 2.0 * np.pi * f
        kappa = (omega**2 * rho * A / (E_complex * I))**0.25

        lam_1 =  kappa
        lam_2 =  1j * kappa
        lam_3 = -kappa
        lam_4 = -1j * kappa

        M = np.array([
            [E_complex*I*lam_1**2 + k_phi*lam_1,
             E_complex*I*lam_2**2 + k_phi*lam_2,
             E_complex*I*lam_3**2 + k_phi*lam_3,
             E_complex*I*lam_4**2 + k_phi*lam_4],
            [E_complex*I*lam_1**3 + k_x,
             E_complex*I*lam_2**3 + k_x,
             E_complex*I*lam_3**3 + k_x,
             E_complex*I*lam_4**3 + k_x],
            [lam_1**2 * np.exp(lam_1 * L),
             lam_2**2 * np.exp(lam_2 * L),
             lam_3**2 * np.exp(lam_3 * L),
             lam_4**2 * np.exp(lam_4 * L)],
            [(E_complex*I*lam_1**3 + m_tip*omega**2) * np.exp(lam_1 * L),
             (E_complex*I*lam_2**3 + m_tip*omega**2) * np.exp(lam_2 * L),
             (E_complex*I*lam_3**3 + m_tip*omega**2) * np.exp(lam_3 * L),
             (E_complex*I*lam_4**3 + m_tip*omega**2) * np.exp(lam_4 * L)],
        ], dtype=np.complex128)

        rhs = np.array([0.0, 0.0, 0.0, -F_tip], dtype=np.complex128)
        c = np.linalg.solve(M, rhs)

        w_x = (c[0]*np.exp(lam_1*x_meas) + c[1]*np.exp(lam_2*x_meas) +
               c[2]*np.exp(lam_3*x_meas) + c[3]*np.exp(lam_4*x_meas))
        v[idx] = 1j * omega * w_x

    return f_hz, v


## Upload experimental data file

Upload your `.npy` or `.npz` file containing the measured FRF.

In [ ]:
uploaded = files.upload()
uploaded_name = list(uploaded.keys())[0]
print("Uploaded:", uploaded_name)


## Experimental data parser

Loads the raw FRF file, handles multiple key naming conventions, removes invalid values, sorts and deduplicates frequencies.

In [ ]:
def _pick_first_key(d, candidates):
    for c in candidates:
        if c in d:
            return c
    return None


def load_experimental_frf(path, x_meas):
    ext = os.path.splitext(path)[1].lower()
    freq_candidates = ["freq", "f", "frequency", "f_hz", "freq_s1", "freq_s2"]
    frf_candidates  = ["frf", "V", "velocity", "response", "H", "s1_fft", "s2_fft"]
    real_candidates = ["real", "V_real", "vr", "Re"]
    imag_candidates = ["imag", "V_imag", "vi", "Im"]

    if ext == ".npz":
        data = np.load(path, allow_pickle=True)
        print("NPZ keys:", list(data.keys()))
        f_key   = _pick_first_key(data, freq_candidates)
        frf_key = _pick_first_key(data, frf_candidates)
        if f_key and frf_key:
            f_hz = np.asarray(data[f_key]).reshape(-1).astype(float)
            V    = np.asarray(data[frf_key]).reshape(-1)
            if not np.iscomplexobj(V):
                raise ValueError(f"FRF key '{frf_key}' is not complex.")
        else:
            real_key = _pick_first_key(data, real_candidates)
            imag_key = _pick_first_key(data, imag_candidates)
            if not (f_key and real_key and imag_key):
                raise ValueError(f"Cannot parse NPZ. Keys: {list(data.keys())}")
            f_hz = np.asarray(data[f_key]).reshape(-1).astype(float)
            V    = np.asarray(data[real_key]).reshape(-1) + 1j * np.asarray(data[imag_key]).reshape(-1)

    elif ext == ".npy":
        arr  = np.load(path, allow_pickle=True)
        data = arr.item() if hasattr(arr, "item") else None
        if not isinstance(data, dict):
            raise ValueError("Plain .npy without frequency info is ambiguous. Use .npz.")
        print("NPY dict keys:", list(data.keys()))
        f_key   = _pick_first_key(data, freq_candidates)
        frf_key = _pick_first_key(data, frf_candidates)
        if f_key and frf_key:
            f_hz = np.asarray(data[f_key]).reshape(-1).astype(float)
            V    = np.asarray(data[frf_key]).reshape(-1)
            if not np.iscomplexobj(V):
                raise ValueError(f"FRF key '{frf_key}' is not complex.")
        else:
            real_key = _pick_first_key(data, real_candidates)
            imag_key = _pick_first_key(data, imag_candidates)
            if not (f_key and real_key and imag_key):
                raise ValueError(f"Cannot parse NPY dict. Keys: {list(data.keys())}")
            f_hz = np.asarray(data[f_key]).reshape(-1).astype(float)
            V    = np.asarray(data[real_key]).reshape(-1) + 1j * np.asarray(data[imag_key]).reshape(-1)
    else:
        raise ValueError("Only .npy and .npz supported.")

    # Validate
    f_hz = np.asarray(f_hz).reshape(-1)
    V    = np.asarray(V).reshape(-1)
    if len(f_hz) != len(V):
        raise ValueError(f"Length mismatch: f_hz={len(f_hz)}, V={len(V)}")

    mask = np.isfinite(f_hz) & np.isfinite(V.real) & np.isfinite(V.imag)
    f_hz, V = f_hz[mask], V[mask]
    mask = f_hz >= 0.0
    f_hz, V = f_hz[mask], V[mask]
    order = np.argsort(f_hz)
    f_hz, V = f_hz[order], V[order]

    df = pd.DataFrame({"f_hz": f_hz, "V_real": V.real, "V_imag": V.imag})
    df = df.groupby("f_hz", as_index=False).mean()
    f_hz = df["f_hz"].to_numpy()
    V    = df["V_real"].to_numpy() + 1j * df["V_imag"].to_numpy()

    x = np.full_like(f_hz, fill_value=x_meas, dtype=float)
    return f_hz, x, V


## Load and filter experimental data

**Frequency cutoff:** We apply a hard cutoff at `MAX_FREQ_HZ` (5000 Hz). Above this:
- The Euler-Bernoulli model is no longer accurate for this beam geometry
- The experimental signal is noise-dominated
- Training on this region actively harms parameter identification

**Minimum frequency:** 0 Hz causes a singular matrix in the forward model, so we start from at least 1 Hz.

In [ ]:
f_exp_hz, x_exp, V_exp = load_experimental_frf(uploaded_name, x_meas=x_meas)

print(f"Loaded {len(f_exp_hz)} points from file")
print(f"f_min = {f_exp_hz.min():.2f} Hz,  f_max = {f_exp_hz.max():.2f} Hz")

# Remove DC / near-zero (singular matrix) and apply upper cutoff
min_freq = max(1.0, f_exp_hz.min())
mask = (f_exp_hz >= min_freq) & (f_exp_hz <= MAX_FREQ_HZ)

f_exp_hz = f_exp_hz[mask]
x_exp    = x_exp[mask]
V_exp    = V_exp[mask]

print(f"After cutoff: {len(f_exp_hz)} points")
print(f"f_min = {f_exp_hz.min():.2f} Hz,  f_max = {f_exp_hz.max():.2f} Hz")


## Smooth and subsample the experimental FRF

**Why smooth?**
The PINN's physics loss wants W(x,ω) to satisfy the beam PDE — a smooth analytical function. Raw experimental FRF contains measurement noise that pulls the data loss in the opposite direction. The optimizer ends up chasing noise instead of physics, and the identified parameters (E0, eta, etc.) never converge cleanly.

Savitzky-Golay filter preserves peak locations and shapes while removing high-frequency noise — it is better than a simple moving average for FRF data.

**Why subsample?**
You have thousands of frequency points but only 5 unknowns (E0, eta, kx, kphi, F). You do not need all of them. Fewer, well-chosen points make training faster and prevent the data loss from numerically swamping the physics loss.

In [ ]:
# Smoothing — Savitzky-Golay filter
# window_length: how many points to use per local polynomial fit (must be odd)
# polyorder: degree of the local polynomial (3 = cubic, preserves peaks well)
# Adjust window_length if you have very few points (must be < len(f_exp_hz))
sg_window = min(51, len(f_exp_hz) // 10 * 2 + 1)  # auto-safe window
sg_window = max(sg_window, 5)

V_mag_smooth   = savgol_filter(np.abs(V_exp),    window_length=sg_window, polyorder=3)
V_phase_smooth = savgol_filter(np.angle(V_exp),  window_length=sg_window, polyorder=3)
V_exp_smooth   = V_mag_smooth * np.exp(1j * V_phase_smooth)

# Subsample to ~300 evenly-spaced points
N_subsample = 300
idx_sub     = np.linspace(0, len(f_exp_hz) - 1, N_subsample, dtype=int)
f_exp_sub   = f_exp_hz[idx_sub]
V_exp_sub   = V_exp_smooth[idx_sub]
x_exp_sub   = x_exp[idx_sub]

print(f"Smoothing window: {sg_window} points")
print(f"Subsampled to {len(f_exp_sub)} points")

# Plot raw vs smoothed to verify
fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axs[0].plot(f_exp_hz, 20*np.log10(np.abs(V_exp) + eps_db), alpha=0.4, label="Raw")
axs[0].plot(f_exp_sub, 20*np.log10(np.abs(V_exp_sub) + eps_db), "r-", lw=1.5, label="Smoothed + subsampled")
axs[0].set_ylabel("Magnitude [dB]")
axs[0].legend()
axs[0].set_title("Experimental FRF: Raw vs Smoothed")

axs[1].plot(f_exp_hz, np.unwrap(np.angle(V_exp)), alpha=0.4, label="Raw")
axs[1].plot(f_exp_sub, np.unwrap(np.angle(V_exp_sub)), "r-", lw=1.5, label="Smoothed + subsampled")
axs[1].set_ylabel("Phase [rad]")
axs[1].set_xlabel("Frequency [Hz]")
axs[1].legend()

plt.tight_layout()
plt.show()


## Generate synthetic FRF

We generate a clean synthetic FRF using the true parameter values. This is used for **Phase 1 validation**: we train the PINN on this clean data first, verify that the identified parameters converge to the true values, and only then switch to experimental data. If the PINN cannot recover parameters from clean synthetic data, there is a bug in the physics implementation — not a data quality problem.

In [ ]:
f_syn_hz, V_syn = euler_beam_changes_forward(
    f_hz=f_exp_sub,   # same frequency grid as subsampled experimental data
    L=L, width=width, height=height, rho=rho,
    E0=E0_true, eta=eta_true, F_tip=F_true,
    m_tip=m_tip, k_phi=kphi_true, k_x=kx_true,
    x_meas=x_meas, bending_axis=bending_axis,
)
print(f"Synthetic data generated: {len(f_syn_hz)} points")


## Choose training data

⚠️ **Always start with `use_synthetic_as_training_data = True`.**

Verify the PINN recovers E0, eta, kx, kphi, F correctly from synthetic data. Once confirmed, set this to `False` to train on your smoothed experimental data.

In [ ]:
# use_synthetic_as_training_data is set in the geometry cell above
if use_synthetic_as_training_data:
    f_train_hz = f_syn_hz.copy()
    x_train    = np.full_like(f_train_hz, x_meas, dtype=float)
    V_train    = V_syn.copy()
    print("Training data = SYNTHETIC")
    print(f"True params: E0={E0_true:.3e}, eta={eta_true:.4f}, kx={kx_true:.3e}, kphi={kphi_true:.3e}, F={F_true:.3f}")
else:
    f_train_hz = f_exp_sub.copy()
    x_train    = x_exp_sub.copy()
    V_train    = V_exp_sub.copy()
    print("Training data = EXPERIMENTAL (smoothed + subsampled)")

omega_train = 2.0 * np.pi * f_train_hz
print(f"N training points: {len(f_train_hz)}")


## Convert to PyTorch tensors and define normalization

**Why normalize?** Neural networks train much faster when inputs are in the range [-1, 1]. Without normalization, the network's linear layers receive inputs spanning orders of magnitude (e.g. omega ranges from ~6 to ~31,000 rad/s), which causes gradient imbalance and slow convergence.

**float64:** All tensors use float64 to ensure accurate 4th-order autograd derivatives in the physics loss.

In [ ]:
# FIX: Use DTYPE=float64 everywhere for accurate higher-order autograd
x_train_t     = torch.tensor(x_train,      dtype=DTYPE, device=device).view(-1, 1)
omega_train_t = torch.tensor(omega_train,  dtype=DTYPE, device=device).view(-1, 1)
Vr_train_t    = torch.tensor(V_train.real, dtype=DTYPE, device=device).view(-1, 1)
Vi_train_t    = torch.tensor(V_train.imag, dtype=DTYPE, device=device).view(-1, 1)

omega_min = float(omega_train.min())
omega_max = float(omega_train.max())

print(f"omega_min = {omega_min:.2f} rad/s")
print(f"omega_max = {omega_max:.2f} rad/s")


def normalize_x(x):
    """Map spatial coordinate [0, L] -> [-1, 1]"""
    return 2.0 * x / L - 1.0


def normalize_omega(omega):
    """Map frequency [omega_min, omega_max] -> [-1, 1]"""
    return 2.0 * (omega - omega_min) / (omega_max - omega_min) - 1.0


## PINN neural network architecture

The network takes (x, ω) as input and outputs (W_real, W_imag) — the real and imaginary parts of the complex displacement field. The velocity FRF is then V = iω·W.

The network is a standard fully-connected network with Tanh activations. Tanh is preferred over ReLU for PINNs because it is infinitely differentiable — ReLU has zero 2nd derivative almost everywhere, which breaks the physics loss entirely.

Note the network is defined with `DTYPE` (float64) weights.

In [ ]:
class PINN(nn.Module):
    """
    Input:  (x, omega)  — spatial position and angular frequency
    Output: (W_real, W_imag) — complex displacement field components
    """
    def __init__(self, layers=[2, 128, 128, 128, 128, 2]):
        super().__init__()
        self.net = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.net.append(nn.Linear(layers[i], layers[i + 1]))
        self.act = nn.Tanh()

    def forward(self, x, omega):
        x_n = normalize_x(x)
        w_n = normalize_omega(omega)
        z   = torch.cat([x_n, w_n], dim=1)

        for layer in self.net[:-1]:
            z = self.act(layer(z))

        out = self.net[-1](z)
        return out[:, 0:1], out[:, 1:2]   # Wr, Wi


# FIX: Cast all network weights to float64
pinn = PINN().to(device).to(DTYPE)
print(pinn)
print(f"Parameters in network: {sum(p.numel() for p in pinn.parameters()):,}")


## Trainable physical parameters

E0, eta, kx, kphi, and F are the quantities we want to identify. They are defined as `nn.Parameter` objects so PyTorch tracks gradients through them.

**Why softplus?** All physical parameters must be positive (negative stiffness or damping is unphysical). Softplus is a smooth approximation to ReLU: `softplus(x) = log(1 + exp(x))`, which maps real numbers to positive values.

**Why scale?** E0 is ~1e9 Pa, kx might be ~1e6 N/m. If we optimize raw values, the gradient for E0 would be ~1000× larger than for kx just due to magnitude, causing imbalance. We divide each parameter by a characteristic scale so all softplus inputs are O(1) numbers.

In [ ]:
# FIX: Parameter scales must match the ORDER OF MAGNITUDE of your expected values.
# The softplus input (the raw parameter) should be O(1) for smooth optimization.
E0_scale    = 1e9    # Pa
eta_scale   = 1.0    # dimensionless
kx_scale    = 1e4    # N/m  (start softer than original 1e6 which was too stiff)
kphi_scale  = 1e2    # N·m/rad
F_scale     = 1.0    # N


def inv_softplus_stable(y):
    """Inverse of softplus: given target value y, find raw input x such that softplus(x)=y"""
    y = np.asarray(y, dtype=np.float64)
    out = np.where(y > 50.0, y, np.log(np.expm1(y)))
    return float(out)


# Dimensionless initial guesses
E0_tilde_init   = E0_init / E0_scale
eta_tilde_init  = eta_init / eta_scale
kx_tilde_init   = kx_init / kx_scale
kphi_tilde_init = kphi_init / kphi_scale
F_tilde_init    = F_init / F_scale

# Raw trainable parameters (in DTYPE=float64)
E0_raw   = nn.Parameter(torch.tensor([inv_softplus_stable(E0_tilde_init)],   dtype=DTYPE, device=device))
eta_raw  = nn.Parameter(torch.tensor([inv_softplus_stable(eta_tilde_init)],  dtype=DTYPE, device=device))
kx_raw   = nn.Parameter(torch.tensor([inv_softplus_stable(kx_tilde_init)],   dtype=DTYPE, device=device))
kphi_raw = nn.Parameter(torch.tensor([inv_softplus_stable(kphi_tilde_init)], dtype=DTYPE, device=device))
F_raw    = nn.Parameter(torch.tensor([inv_softplus_stable(F_tilde_init)],    dtype=DTYPE, device=device))

trainable_params = list(pinn.parameters()) + [E0_raw, eta_raw, kx_raw, kphi_raw, F_raw]

# FIX: Lower initial LR for physical parameters to avoid large jumps early in training.
# We use param groups: slightly lower LR for the scalar params vs network weights.
optimizer = torch.optim.Adam([
    {"params": list(pinn.parameters()), "lr": 1e-3},
    {"params": [E0_raw, eta_raw, kx_raw, kphi_raw, F_raw], "lr": 5e-4},
])

# FIX: Learning rate scheduler — reduce LR when loss plateaus.
# This lets training make large progress early and fine-tune later.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=1000, factor=0.5, min_lr=1e-7, verbose=True
)

print("Initial parameter estimates:")
print(f"  E0_tilde_init   = {E0_tilde_init:.4f}  (raw = {inv_softplus_stable(E0_tilde_init):.4f})")
print(f"  eta_tilde_init  = {eta_tilde_init:.4f}  (raw = {inv_softplus_stable(eta_tilde_init):.4f})")
print(f"  kx_tilde_init   = {kx_tilde_init:.4f}  (raw = {inv_softplus_stable(kx_tilde_init):.4f})")
print(f"  kphi_tilde_init = {kphi_tilde_init:.4f}  (raw = {inv_softplus_stable(kphi_tilde_init):.4f})")
print(f"  F_tilde_init    = {F_tilde_init:.4f}  (raw = {inv_softplus_stable(F_tilde_init):.4f})")


## Collocation points for PDE residual

Collocation points are locations in the (x, ω) domain where we evaluate the PDE residual. The PINN must satisfy `EI·W_xxxx - ρA·ω²·W = 0` at all these points — this is what enforces the physics.

We use a 2D grid covering the full spatial domain [0, L] and the frequency range [ω_min, ω_max]. These points are separate from the data points — they enforce the governing equation everywhere in the domain, not just at measurement locations.

**Note:** `x_col` needs `requires_grad=True` because we differentiate W up to 4 times with respect to x.

In [ ]:
# FIX: Use more collocation points for better PDE coverage.
# 50 x 50 = 2500 points covers the domain well without excessive memory use.
Nx = 50
Nw = 50

x_c     = torch.linspace(0.0, L, Nx, device=device, dtype=DTYPE).view(-1, 1)
omega_c = torch.linspace(omega_min, omega_max, Nw, device=device, dtype=DTYPE).view(-1, 1)

xx, ww  = torch.meshgrid(x_c.squeeze(), omega_c.squeeze(), indexing="ij")
x_col   = xx.reshape(-1, 1).clone().detach().requires_grad_(True)
omega_col = ww.reshape(-1, 1).clone().detach()   # no grad needed w.r.t. omega

print(f"Collocation grid: {Nx} x {Nw} = {x_col.shape[0]} points")


## Physics utilities

`grad(y, x)` computes dy/dx using PyTorch autograd. We call it four times in sequence to get the 4th spatial derivative W_xxxx needed for the Euler-Bernoulli PDE.

`positive_params()` retrieves the current physical parameter values from their raw forms, applying softplus and the physical scale. This is called every training step.

In [ ]:
def grad(y, x):
    """Compute dy/dx using autograd, keeping the graph for higher-order derivatives."""
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def positive_params():
    """Return physical parameters as positive tensors in correct physical units."""
    sp = torch.nn.functional.softplus
    E0   = (sp(E0_raw)   + 1e-12) * E0_scale
    eta  = (sp(eta_raw)  + 1e-12) * eta_scale
    kx   = (sp(kx_raw)   + 1e-12) * kx_scale
    kphi = (sp(kphi_raw) + 1e-12) * kphi_scale
    F    = (sp(F_raw)    + 1e-12) * F_scale
    return E0, eta, kx, kphi, F


## Loss function

The total loss has three components:

1. **PDE loss** — penalizes violation of the Euler-Bernoulli equation at collocation points. This enforces the beam physics everywhere in the domain.

2. **BC loss** — penalizes violation of the 4 boundary conditions (2 at x=0, 2 at x=L). This is weighted highest because BCs are exact constraints with no noise.

3. **Data loss** — penalizes the difference between the PINN's predicted velocity FRF and the measured data at the training frequencies.

**Why V = iω·W?** The FRF you measured is velocity (V = dx/dt). In frequency domain, differentiation is multiplication by iω. So velocity = iω × displacement → V_real = -ω·W_imag, V_imag = +ω·W_real.

**Loss weights:**
- `w_pde = 10` — physics dominates
- `w_bc = 100` — BCs are hard constraints
- `w_data = 1` — data is trusted but noisy

In [ ]:
# Fixed geometry tensors (float64)
A_t     = torch.tensor(A,      dtype=DTYPE, device=device)
I_t     = torch.tensor(I,      dtype=DTYPE, device=device)
rho_t   = torch.tensor(rho,    dtype=DTYPE, device=device)
m_tip_t = torch.tensor(m_tip,  dtype=DTYPE, device=device)


def pinn_loss(
    pinn, x_col, omega_col,
    x_data, omega_data, Vr_data, Vi_data,
    w_pde=10.0, w_bc=100.0, w_data=1.0,
):
    E0, eta, kx, kphi, F = positive_params()

    # Complex bending stiffness: EI = E0*(1+i*eta)*I
    EI_r = E0 * I_t            # real part
    EI_i = E0 * eta * I_t      # imaginary part (hysteretic damping)

    # ── PDE residual: EI·W_xxxx - rho*A*omega^2*W = 0 ──────────────────────
    Wr, Wi = pinn(x_col, omega_col)

    Wr_x    = grad(Wr, x_col);  Wi_x    = grad(Wi, x_col)
    Wr_xx   = grad(Wr_x, x_col); Wi_xx  = grad(Wi_x, x_col)
    Wr_xxx  = grad(Wr_xx, x_col); Wi_xxx = grad(Wi_xx, x_col)
    Wr_xxxx = grad(Wr_xxx, x_col); Wi_xxxx = grad(Wi_xxx, x_col)

    omega2 = omega_col**2

    # Real and imaginary parts of EI*W_xxxx (complex multiplication)
    EI_W4_r = EI_r * Wr_xxxx - EI_i * Wi_xxxx
    EI_W4_i = EI_r * Wi_xxxx + EI_i * Wr_xxxx

    r_pde_r = EI_W4_r - rho_t * A_t * omega2 * Wr
    r_pde_i = EI_W4_i - rho_t * A_t * omega2 * Wi

    pde_loss = torch.mean(r_pde_r**2 + r_pde_i**2)

    # ── Boundary conditions ──────────────────────────────────────────────────
    # Convention (beam_theories_changes.py):
    #   x=0: EI*w''(0) + kphi*w'(0) = 0    [rotational spring at base]
    #   x=0: EI*w'''(0) + kx*w(0)   = 0    [translational spring at base]
    #   x=L: w''(L) = 0                     [free end, zero moment]
    #   x=L: EI*w'''(L) + m_tip*omega^2*w(L) + F = 0   [tip mass + force]
    omega_b = omega_col.detach()

    # BCs at x = 0
    x0       = torch.zeros_like(omega_b, requires_grad=True)
    Wr0, Wi0 = pinn(x0, omega_b)

    Wr0_x   = grad(Wr0, x0);  Wi0_x   = grad(Wi0, x0)
    Wr0_xx  = grad(Wr0_x, x0); Wi0_xx  = grad(Wi0_x, x0)
    Wr0_xxx = grad(Wr0_xx, x0); Wi0_xxx = grad(Wi0_xx, x0)

    bc0_m_r = (EI_r * Wr0_xx  - EI_i * Wi0_xx)  + kphi * Wr0_x
    bc0_m_i = (EI_r * Wi0_xx  + EI_i * Wr0_xx)  + kphi * Wi0_x
    bc0_v_r = (EI_r * Wr0_xxx - EI_i * Wi0_xxx) + kx * Wr0
    bc0_v_i = (EI_r * Wi0_xxx + EI_i * Wr0_xxx) + kx * Wi0

    # BCs at x = L
    xL       = torch.full_like(omega_b, fill_value=L, requires_grad=True)
    WrL, WiL = pinn(xL, omega_b)

    WrL_x   = grad(WrL, xL);  WiL_x   = grad(WiL, xL)
    WrL_xx  = grad(WrL_x, xL); WiL_xx  = grad(WiL_x, xL)
    WrL_xxx = grad(WrL_xx, xL); WiL_xxx = grad(WiL_xx, xL)

    bcL_m_r = WrL_xx
    bcL_m_i = WiL_xx
    bcL_v_r = (EI_r * WrL_xxx - EI_i * WiL_xxx) + m_tip_t * (omega_b**2) * WrL + F
    bcL_v_i = (EI_r * WiL_xxx + EI_i * WrL_xxx) + m_tip_t * (omega_b**2) * WiL

    bc_loss = torch.mean(
        bc0_m_r**2 + bc0_m_i**2 +
        bc0_v_r**2 + bc0_v_i**2 +
        bcL_m_r**2 + bcL_m_i**2 +
        bcL_v_r**2 + bcL_v_i**2
    )

    # ── Data loss: V = i*omega*W → V_r = -omega*Wi, V_i = omega*Wr ─────────
    Wr_d, Wi_d = pinn(x_data, omega_data)
    Vp_r = -omega_data * Wi_d
    Vp_i =  omega_data * Wr_d

    data_loss = torch.mean((Vp_r - Vr_data)**2 + (Vp_i - Vi_data)**2)

    total = w_pde * pde_loss + w_bc * bc_loss + w_data * data_loss

    logs = {
        "loss": total.detach().item(),
        "pde":  pde_loss.detach().item(),
        "bc":   bc_loss.detach().item(),
        "data": data_loss.detach().item(),
        "E0":   E0.detach().item(),
        "eta":  eta.detach().item(),
        "kx":   kx.detach().item(),
        "kphi": kphi.detach().item(),
        "F":    F.detach().item(),
    }
    return total, logs


## Training loop

**Why 30,000 epochs?** PINNs with 4th-order PDEs and simultaneous parameter identification converge much more slowly than standard supervised learning. At 5,000 epochs the physics residuals are typically still large. 30k is a reasonable minimum; run longer if the loss is still decreasing.

**Loss weights used:**
- `w_pde = 10` — we trust the physics more than the (noisy) data
- `w_bc = 100` — boundary conditions are exact constraints
- `w_data = 1` — data is informative but contains noise

The scheduler automatically halves the learning rate when the total loss stops improving for 1000 epochs, enabling fine-grained convergence in later stages.

In [ ]:
# FIX: Increased epochs from 5000 to 30000.
# PINNs with 4th-order PDEs + parameter identification need long training.
epochs = 30000

# FIX: Rebalanced weights — physics (pde + bc) dominates over data.
# This is critical when training on noisy experimental data.
w_pde  = 10.0
w_bc   = 100.0
w_data = 1.0

history = {k: [] for k in ["loss","pde","bc","data","E0","eta","kx","kphi","F"]}

for epoch in range(epochs):
    optimizer.zero_grad()

    loss, logs = pinn_loss(
        pinn=pinn,
        x_col=x_col, omega_col=omega_col,
        x_data=x_train_t, omega_data=omega_train_t,
        Vr_data=Vr_train_t, Vi_data=Vi_train_t,
        w_pde=w_pde, w_bc=w_bc, w_data=w_data,
    )

    loss.backward()
    optimizer.step()
    scheduler.step(logs["loss"])

    for k in history:
        history[k].append(logs[k])

    if epoch % 500 == 0:
        print(
            f"Epoch {epoch:6d} | Loss {logs['loss']:.3e} | "
            f"PDE {logs['pde']:.3e} | BC {logs['bc']:.3e} | Data {logs['data']:.3e} | "
            f"E0 {logs['E0']:.4e} | eta {logs['eta']:.5f} | "
            f"kx {logs['kx']:.3e} | kphi {logs['kphi']:.3e} | F {logs['F']:.4f}"
        )

print("\nTraining complete.")


## Training history plots

All three loss terms should decrease and then plateau. If the **data loss is stuck high** while pde and bc are low, the physics model may not be a good fit for your beam. If **pde loss stays high**, the network is struggling to satisfy the PDE — try more collocation points or longer training.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

axs[0,0].semilogy(history["loss"],  label="total")
axs[0,0].semilogy(history["pde"],   label="pde")
axs[0,0].semilogy(history["bc"],    label="bc")
axs[0,0].semilogy(history["data"],  label="data")
axs[0,0].legend(); axs[0,0].set_title("Loss history"); axs[0,0].set_xlabel("Epoch")

axs[0,1].plot(history["E0"])
axs[0,1].set_title("E0 convergence [Pa]"); axs[0,1].set_xlabel("Epoch")
if use_synthetic_as_training_data:
    axs[0,1].axhline(E0_true, color="r", linestyle="--", label=f"True={E0_true:.2e}")
    axs[0,1].legend()

axs[1,0].plot(history["eta"], label="eta")
axs[1,0].plot(history["F"],   label="F")
axs[1,0].legend(); axs[1,0].set_title("eta and F"); axs[1,0].set_xlabel("Epoch")
if use_synthetic_as_training_data:
    axs[1,0].axhline(eta_true, color="r", linestyle="--", label=f"True eta={eta_true}")

axs[1,1].plot(history["kx"],   label="kx")
axs[1,1].plot(history["kphi"], label="kphi")
axs[1,1].legend(); axs[1,1].set_title("kx and kphi"); axs[1,1].set_xlabel("Epoch")

plt.tight_layout()
plt.show()


## Final identified parameters

These are the physical parameters identified by the PINN after training. When training on synthetic data, compare these to the true values to verify correctness. When training on experimental data, these are your beam's estimated material and boundary parameters.

In [ ]:
E0_est, eta_est, kx_est, kphi_est, F_est = positive_params()

print("\n=== FINAL IDENTIFIED PARAMETERS ===")
print(f"E0   = {E0_est.item():.6e} Pa")
print(f"eta  = {eta_est.item():.6f}")
print(f"kx   = {kx_est.item():.6e} N/m")
print(f"kphi = {kphi_est.item():.6e} N*m/rad")
print(f"F    = {F_est.item():.6f} N")

if use_synthetic_as_training_data:
    print("\n=== COMPARISON WITH TRUE VALUES ===")
    print(f"E0:   identified = {E0_est.item():.4e},  true = {E0_true:.4e},  error = {abs(E0_est.item()-E0_true)/E0_true*100:.2f}%")
    print(f"eta:  identified = {eta_est.item():.5f},  true = {eta_true:.5f},  error = {abs(eta_est.item()-eta_true)/eta_true*100:.2f}%")
    print(f"kx:   identified = {kx_est.item():.4e},  true = {kx_true:.4e},  error = {abs(kx_est.item()-kx_true)/kx_true*100:.2f}%")
    print(f"kphi: identified = {kphi_est.item():.4e},  true = {kphi_true:.4e},  error = {abs(kphi_est.item()-kphi_true)/kphi_true*100:.2f}%")
    print(f"F:    identified = {F_est.item():.5f},  true = {F_true:.5f},  error = {abs(F_est.item()-F_true)/F_true*100:.2f}%")


## Predict FRF from trained PINN

We evaluate the PINN at the training frequencies to get the predicted FRF. The velocity is reconstructed from the displacement field: V = iω·W → V_real = -ω·W_imag, V_imag = ω·W_real.

In [ ]:
pinn.eval()

with torch.no_grad():
    x_eval_t  = torch.full_like(omega_train_t, fill_value=x_meas, dtype=DTYPE)
    Wr_pred, Wi_pred = pinn(x_eval_t, omega_train_t)

    Vp_r = (-omega_train_t * Wi_pred).cpu().numpy().reshape(-1)
    Vp_i = ( omega_train_t * Wr_pred).cpu().numpy().reshape(-1)
    V_pred = Vp_r + 1j * Vp_i

f_plot_hz = f_train_hz.copy()
print(f"Prediction generated for {len(f_plot_hz)} frequency points")


## PINN prediction vs training data

This is the key diagnostic plot. The PINN output should follow the **smooth envelope** of the training data — matching the resonance peaks and the broad magnitude trend. It will not (and should not) fit the noise between peaks. If the curves are completely misaligned, revisit the loss history and parameter convergence plots.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

if plot_in_db:
    axs[0].plot(f_plot_hz, 20*np.log10(np.abs(V_train) + eps_db), alpha=0.6, label="Training data")
    axs[0].plot(f_plot_hz, 20*np.log10(np.abs(V_pred)  + eps_db), "r--", lw=2, label="PINN")
    axs[0].set_ylabel("Magnitude [dB]")
else:
    axs[0].plot(f_plot_hz, np.abs(V_train), alpha=0.6, label="Training data")
    axs[0].plot(f_plot_hz, np.abs(V_pred),  "r--", lw=2, label="PINN")
    axs[0].set_ylabel("Magnitude")

axs[0].legend(); axs[0].set_title("PINN vs Training FRF")

axs[1].plot(f_plot_hz, np.unwrap(np.angle(V_train)), alpha=0.6, label="Training data")
axs[1].plot(f_plot_hz, np.unwrap(np.angle(V_pred)),  "r--", lw=2, label="PINN")
axs[1].set_ylabel("Phase [rad]")
axs[1].set_xlabel("Frequency [Hz]")
axs[1].legend()

plt.tight_layout()
plt.show()


## If trained on synthetic: compare PINN vs experimental data

This cell only runs if you trained on synthetic data. It shows how well the PINN (with identified parameters from synthetic) generalizes to the experimental data — useful for checking if the forward model is a reasonable fit for the real beam.

In [ ]:
if use_synthetic_as_training_data:
    # Evaluate PINN on experimental frequency grid
    omega_exp_t = torch.tensor(2*np.pi*f_exp_sub, dtype=DTYPE, device=device).view(-1, 1)
    x_exp_t     = torch.full_like(omega_exp_t, fill_value=x_meas)

    with torch.no_grad():
        Wr_p, Wi_p = pinn(x_exp_t, omega_exp_t)
        Vp_r_exp = (-omega_exp_t * Wi_p).cpu().numpy().reshape(-1)
        Vp_i_exp = ( omega_exp_t * Wr_p).cpu().numpy().reshape(-1)
        V_pred_exp = Vp_r_exp + 1j * Vp_i_exp

    fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axs[0].plot(f_exp_sub, 20*np.log10(np.abs(V_exp_sub) + eps_db), alpha=0.6, label="Experimental (smoothed)")
    axs[0].plot(f_exp_sub, 20*np.log10(np.abs(V_pred_exp) + eps_db), "r--", lw=2, label="PINN (from synthetic params)")
    axs[0].set_ylabel("Magnitude [dB]")
    axs[0].legend(); axs[0].set_title("Experimental vs PINN (trained on synthetic)")

    axs[1].plot(f_exp_sub, np.unwrap(np.angle(V_exp_sub)), alpha=0.6, label="Experimental (smoothed)")
    axs[1].plot(f_exp_sub, np.unwrap(np.angle(V_pred_exp)), "r--", lw=2, label="PINN")
    axs[1].set_ylabel("Phase [rad]")
    axs[1].set_xlabel("Frequency [Hz]")
    axs[1].legend()

    plt.tight_layout()
    plt.show()


## Save identified parameters and prediction

Saves the identified parameters to JSON and the full FRF prediction to CSV. Run this after each experiment to build your dataset of identified beam parameters.

In [ ]:
results = {
    "E0_est":   float(E0_est.item()),
    "eta_est":  float(eta_est.item()),
    "kx_est":   float(kx_est.item()),
    "kphi_est": float(kphi_est.item()),
    "F_est":    float(F_est.item()),
    "L": L, "width": width, "height": height, "rho": rho,
    "m_tip": m_tip, "x_meas": x_meas, "bending_axis": bending_axis,
    "MAX_FREQ_HZ": MAX_FREQ_HZ,
    "trained_on": "synthetic" if use_synthetic_as_training_data else "experimental",
}

with open("pinn_v2_identified_params.json", "w") as f:
    json.dump(results, f, indent=2)

df_pred = pd.DataFrame({
    "f_hz":       f_plot_hz,
    "V_train_real": V_train.real,
    "V_train_imag": V_train.imag,
    "V_pred_real":  V_pred.real,
    "V_pred_imag":  V_pred.imag,
})
df_pred.to_csv("pinn_v2_prediction.csv", index=False)

print("Saved: pinn_v2_identified_params.json")
print("Saved: pinn_v2_prediction.csv")
print(json.dumps(results, indent=2))
